# 06 - Explainable AI (XAI)

Using SHAP to explain the selected XGBoost model — both globally (which features drive predictions overall) and locally (why the model made a specific prediction for one patient).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import joblib
import shap

DATA_PATH = Path("../data/processed")
MODEL_PATH = Path("../outputs/models")
FIG_PATH = Path("../outputs/figures")
FIG_PATH.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_PATH / "train_processed.csv")
test = pd.read_csv(DATA_PATH / "test_processed.csv")
X_train, y_train = train.drop(columns="Outcome"), train["Outcome"]
X_test, y_test = test.drop(columns="Outcome"), test["Outcome"]

# Explain whichever model notebook 04 actually selected — SHAP TreeExplainer works for
# both XGBoost and the K-Means + Random Forest candidate.
selection_metadata = joblib.load(MODEL_PATH / "selection_metadata.pkl")
selected_name = selection_metadata["selected_model"]
print("Explaining selected model:", selected_name)

if selected_name.startswith("XGBoost"):
    model = joblib.load(MODEL_PATH / "xgboost_final_model.pkl")
    X_explain = X_test
else:
    model = joblib.load(MODEL_PATH / "kmeans_rf_final_model.pkl")
    X_explain = pd.read_csv(DATA_PATH / "X_test_kmeans_features.csv")

best_xgb = model  # kept as best_xgb for compatibility with the rest of this notebook
X_test = X_explain


**Build SHAP explainer (TreeExplainer — exact and fast for XGBoost)**

In [ ]:
explainer = shap.TreeExplainer(best_xgb)
shap_values = explainer.shap_values(X_test)


**Global feature importance — SHAP summary (beeswarm)**

In [ ]:
shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout()
plt.savefig(FIG_PATH / "shap_summary_beeswarm.png", dpi=120, bbox_inches="tight")
plt.show()


**Global feature importance — mean |SHAP value| bar chart**

In [ ]:
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig(FIG_PATH / "shap_summary_bar.png", dpi=120, bbox_inches="tight")
plt.show()


**Reading the global explanation**

Expect Glucose (and Glucose-derived engineered features) to dominate — consistent with clinical knowledge (glucose is the direct diagnostic marker) and with the EDA notebook's distribution plots. BMI and Age typically rank next. If an engineered feature like `Glucose_BMI` ranks above its parent features, that's evidence the interaction term earned its place rather than just adding redundant noise — worth confirming here rather than assuming.

**Local explanation — one correctly predicted diabetic patient**

In [ ]:
y_pred = best_xgb.predict(X_test)
idx_candidates = X_test.index[(y_test.values == 1) & (y_pred == 1)]
idx = idx_candidates[0]

shap.force_plot(
    explainer.expected_value,
    shap_values[idx],
    X_test.iloc[idx],
    matplotlib=True,
    show=False
)
plt.tight_layout()
plt.savefig(FIG_PATH / "shap_force_plot_example.png", dpi=120, bbox_inches="tight")
plt.show()

print("Patient row:")
X_test.iloc[[idx]]


**Local explanation — waterfall view (often clearer than force plot)**

In [ ]:
shap_exp = shap.Explanation(
    values=shap_values[idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[idx],
    feature_names=X_test.columns.tolist()
)
shap.plots.waterfall(shap_exp, show=False)
plt.tight_layout()
plt.savefig(FIG_PATH / "shap_waterfall_example.png", dpi=120, bbox_inches="tight")
plt.show()


**Local explanation — clinical-style table (same logic used in the deployed Streamlit app)**

The force plot and waterfall plot above are useful for a technical audience, but the deployed app presents this same SHAP output as a sortable table with plain-language clinical context, since that's more readable for a non-technical reader (e.g. a supervisor skimming the write-up, or a clinician). Reproducing that exact logic here keeps the notebook and the app consistent.

In [ ]:
import numpy as np

CLINICAL_CONTEXT = {
    "Pregnancies": "Number of pregnancies — repeated pregnancies are a modest, non-modifiable risk marker.",
    "Glucose": "Plasma glucose concentration — the single strongest direct clinical marker for diabetes.",
    "BloodPressure": "Diastolic blood pressure — hypertension often co-occurs with metabolic risk.",
    "SkinThickness": "Triceps skinfold thickness — a rough proxy for subcutaneous body fat.",
    "Insulin": "2-hour serum insulin — abnormal levels reflect insulin resistance or deficiency.",
    "BMI": "Body Mass Index — a core, well-established risk factor for type 2 diabetes.",
    "DiabetesPedigreeFunction": "A composite score of family history / genetic predisposition to diabetes.",
    "Age": "Age in years — diabetes risk rises with age, especially past 45.",
    "Cluster_Dist_1": "Distance to patient cluster 1 centroid (K-Means-derived similarity feature).",
    "Cluster_Dist_2": "Distance to patient cluster 2 centroid (K-Means-derived similarity feature).",
}


def explain_prediction(model, X_row, prob, explainer=None):
    """
    Generate a per-patient SHAP explanation table + plain-language narrative.
    Same function used in streamlit_app.py so the notebook and deployed app agree.

    model     : trained tree-based model (XGBoost / RandomForest)
    X_row     : single-row DataFrame of the patient's (preprocessed) features
    prob      : model's predicted probability of diabetes for this patient
    explainer : optional pre-built shap.TreeExplainer (reused here to avoid rebuilding it)
    """
    if explainer is None:
        explainer = shap.TreeExplainer(model)
    row_shap_values = explainer.shap_values(X_row)

    if isinstance(row_shap_values, list):
        values_row = row_shap_values[1][0]
        base_value = explainer.expected_value[1]
    else:
        values_row = row_shap_values[0]
        base_value = explainer.expected_value
        if isinstance(base_value, (list, np.ndarray)):
            base_value = base_value[-1]

    explain_df = pd.DataFrame({
        "Feature": X_row.columns,
        "Patient Value": X_row.iloc[0].values,
        "Contribution": np.asarray(values_row, dtype=float),
    })
    explain_df["Direction"] = np.where(
        explain_df["Contribution"] > 0, "↑ Increases risk", "↓ Decreases risk"
    )
    explain_df["Clinical context"] = explain_df["Feature"].map(CLINICAL_CONTEXT).fillna("Engineered feature.")
    explain_df["Abs"] = explain_df["Contribution"].abs()
    explain_df = explain_df.sort_values("Abs", ascending=False).drop(columns="Abs")

    top = explain_df.head(3)
    increasers = top[top["Contribution"] > 0]["Feature"].tolist()
    decreasers = top[top["Contribution"] < 0]["Feature"].tolist()

    narrative_parts = []
    if increasers:
        narrative_parts.append(f"The strongest factors **pushing risk up** were {', '.join(increasers)}.")
    if decreasers:
        narrative_parts.append(f"The strongest factors **pushing risk down** were {', '.join(decreasers)}.")
    narrative_parts.append(
        f"Starting from a baseline risk of {base_value:.1%} across the training population, "
        f"these feature values combined to move the prediction to {prob:.1%} for this patient."
    )
    return explain_df, " ".join(narrative_parts)


# Reuse the same patient (idx) as the force plot / waterfall examples above
prob_idx = model.predict_proba(X_test.iloc[[idx]])[0, 1]
explain_df, narrative = explain_prediction(model, X_test.iloc[[idx]], prob_idx, explainer=explainer)

display(explain_df.style.format({"Patient Value": "{:.2f}", "Contribution": "{:+.4f}"}))
print()
print(narrative)

**Dependence plot — how Glucose drives predictions, and its interaction with BMI**

In [ ]:
shap.dependence_plot("Glucose", shap_values, X_test, interaction_index="BMI", show=False)
plt.tight_layout()
plt.savefig(FIG_PATH / "shap_dependence_glucose_bmi.png", dpi=120, bbox_inches="tight")
plt.show()


**XAI summary for the write-up**

- The model's top predictors align with clinical domain knowledge (Glucose > BMI/Age > others) — this is a sanity check the paper's 96%+ claim doesn't offer, since perfect/near-perfect accuracy from leakage gives no genuine signal to interpret.
- SHAP force/waterfall plots let you show a supervisor *why* an individual patient was flagged, not just an aggregate score — useful for the "clinical decision support" angle the paper claims but never demonstrates.
- This closes the loop for the case study: paper claims 96.6% with no leakage check or interpretability shown → your honest pipeline gets ~75-80% with documented preprocessing, threshold justification, and explainability at the patient level.